In [ ]:

import os, sys, glob, shutil, time, json, gc
import numpy as np, pandas as pd
t0=time.perf_counter()
def log(m): print(f"[{time.perf_counter()-t0:6.0f}с] {m}", flush=True)
base=os.path.dirname(glob.glob("/kaggle/input/**/items_human.parquet", recursive=True)[0])
prev=os.path.dirname(glob.glob("/kaggle/input/**/features_human.npy", recursive=True)[0])
vald=os.path.dirname(glob.glob("/kaggle/input/**/llm_valid_pairs.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/src",exist_ok=True)
code_dir=os.path.dirname(glob.glob("/kaggle/input/**/pair_features.py", recursive=True)[0])
for p in glob.glob(code_dir+"/*.py"): shutil.copy(p,"/kaggle/working/src/")
open("/kaggle/working/src/__init__.py","a").close()
os.makedirs("/kaggle/working/models",exist_ok=True)
shutil.copy(code_dir+"/anti_words.json","/kaggle/working/models/anti_words.json")
os.chdir("/kaggle/working"); sys.path.insert(0,"/kaggle/working")
from src.pair_features import build_matrix, feature_names
from src.hybrid import product_disjoint_pair_masks
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

pairs=pd.read_parquet(vald+"/llm_valid_pairs.parquet")
items=pd.read_parquet(vald+"/llm_valid_items.parquet")
log(f"отложенный LLM-фолд: {len(pairs):,} пар, {len(items):,} карточек, доля+ {pairs['label'].mean():.3f}")

t=time.perf_counter()
E=build_matrix(items,pairs,with_neighbours=True)
log(f"признаки посчитаны: {E.shape} за {time.perf_counter()-t:.0f}с")
np.save("/kaggle/working/llmval_features.npy",E)

names=feature_names(True); NB=[i for i,n in enumerate(names) if n.startswith("nb_")]
keep=[i for i in range(len(names)) if i not in NB]
Xh=np.load(prev+"/features_human.npy"); Xl=np.load(prev+"/features_llm.npy")
hm=pd.read_parquet(base+"/matches.parquet",columns=["id1","id2","target"])
lp=pd.read_parquet(base+"/llm_pairs_sel.parquet")
yh=hm["target"].to_numpy(np.int8); yl=lp["label"].to_numpy(np.int8)
tm,vm=product_disjoint_pair_masks(hm["id1"].to_numpy(),hm["id2"].to_numpy(),0,3)
rel=np.flatnonzero(~vm)
P=dict(max_iter=800,learning_rate=0.05,max_leaf_nodes=63,random_state=0,early_stopping=False)
X=np.vstack([Xl,Xh[rel]]); yy=np.concatenate([yl,yh[rel]])
full=HistGradientBoostingClassifier(**P).fit(X,yy)
nonb=HistGradientBoostingClassifier(**P).fit(X[:,keep],yy)
log("модели обучены")

y=pairs["label"].to_numpy(np.int8)
cat=items.set_index("id")["category"].astype(str)
c=pairs["id1"].map(cat).fillna("?").astype(str).to_numpy()
def macro(p): return float(np.mean([average_precision_score(y[c==k],p[c==k])
    for k in np.unique(c) if len(np.unique(y[c==k]))>1 and (c==k).sum()>200]))
pf=full.predict_proba(E)[:,1]; pn=nonb.predict_proba(E[:,keep])[:,1]
np.save("/kaggle/working/llmval_pred_full.npy",pf); np.save("/kaggle/working/llmval_pred_noneigh.npy",pn)
log(f"признаки со 128:        {macro(pf):.6f}")
log(f"признаки без окрестн.:  {macro(pn):.6f}")
log("ВАЖНО: негативы здесь от ретривала организаторов, а не наши синтетические")
log("готово")
